# Set Up Environment

## Import Libraries

In [ ]:
import pandas as pd
import geopandas as gpd

## Set Environment Variables

In [ ]:
PROJECT_CRS = "EPSG:3566"

# Input Data

## Preprocessed Forecast Results

In [ ]:
forecast_results = pd.read_csv('results/final_forecast_df.csv')
forecast_results[['externalid', 'year', 'final_forecast']]

## Traffic Factors

In [ ]:
gdf_master_segments = gpd.read_file(
    "zip://data/updated-traffic-factors/Master_Segs_withFactors_20251120.zip"
).to_crs(PROJECT_CRS)

gdf_master_segments

# Process Data

## Interpolate Final Forecast Values

In [ ]:
# --- 1. Define the full range of years ---
min_year = forecast_results['year'].min()
max_year = forecast_results['year'].max()
all_years = range(min_year, max_year + 1)

In [ ]:
# --- 2. Prepare the full index DataFrame (externalid x year) ---
unique_ids = forecast_results['externalid'].unique()

full_index_df = pd.MultiIndex.from_product(
    [unique_ids, all_years],
    names=['externalid', 'year']
).to_frame().reset_index(drop=True)

In [ ]:
# --- 3. Merge the original data onto the full index ---
# This creates a DataFrame with all years, where the new rows have NaN in 'final_forecast'
merged_df = full_index_df.merge(forecast_results, on=['externalid', 'year'], how='left')

In [ ]:
# --- 4. Interpolate the NaN values (Linear Interpolation) ---
# The result is a Series perfectly aligned with the rows in merged_df
interpolated_values_series = merged_df.groupby('externalid')['final_forecast'].apply(
    lambda x: x.interpolate(method='linear')
)

In [ ]:
# --- 5. Assign the new column and perform cleanup ---
# Create the new column 'interpolated_forecast' using the series values
merged_df['interpolated_forecast'] = interpolated_values_series.values

In [ ]:
# Round the new forecasts to the nearest integer and convert to int
merged_df['interpolated_forecast'] = merged_df['interpolated_forecast'].round().astype(int)

# The original 'final_forecast' column is still present (with NaNs for interpolated years)
final_df_with_both = merged_df.copy()

print("--- Final DataFrame with Both Columns ---")
print(final_df_with_both.head(15))

--- Final DataFrame with Both Columns ---
    externalid  year    PROJ_GRP      linear_forecast_notes       segid  \
0         3601  2027  Since 2003  as early as there is data  1082_000.0   
1         3601  2028         NaN                        NaN         NaN   
2         3601  2029         NaN                        NaN         NaN   
3         3601  2030         NaN                        NaN         NaN   
4         3601  2031         NaN                        NaN         NaN   
5         3601  2032  Since 2003  as early as there is data  1082_000.0   
6         3601  2033         NaN                        NaN         NaN   
7         3601  2034         NaN                        NaN         NaN   
8         3601  2035         NaN                        NaN         NaN   
9         3601  2036  Since 2003  as early as there is data  1082_000.0   
10        3601  2037         NaN                        NaN         NaN   
11        3601  2038         NaN                        Na